# Hold’em opponent history and decision integration

We connect Meeting 4’s Leduc opponent-history method to Meeting 2’s decision model.
Pluribus provides repeated-player histories; PokerBench remains the original decision benchmark.
This is a **128 training / 64 validation decision integration check** on observed human actions,
not an evaluation of optimal play. Both models receive the same current situation; the
history-aware versions also receive completed-hand opponent summaries.

Local setup: `uv sync --locked --extra analysis --extra ml --extra experiments`, then open this
notebook in that environment. Colab installs the project below. Use a checkout containing
the Hold’em integration workflow.
All replay, sampling, forecasting, and training logic lives in the package; this notebook
and `python -m poker_decision.holdem_integration STEP` call the same functions.


In [ ]:
import runpy
import subprocess
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/poker_decision").is_dir()), None
)
if ROOT is None and Path("/content").is_dir():
    ROOT = Path("/content/Poker-Decision-Model")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/nmokey/Poker-Decision-Model.git", str(ROOT)],
            check=True,
        )
if ROOT is None or not (ROOT / "src/poker_decision/holdem_integration.py").is_file():
    raise RuntimeError("Open a checkout containing the Hold'em integration.")
sys.path.insert(0, str(ROOT / "src"))
setup = runpy.run_path(
    str(ROOT / "scripts/notebook_setup.py"),
    init_globals={"ROOT": ROOT, "EXTRAS": "analysis,ml,experiments"},
)

In [ ]:
from poker_decision.holdem_integration import DEFAULT_OUT, main

OUT = DEFAULT_OUT
COMMON = ["--out", str(OUT)]
print("Outputs:", OUT)

## 1. Prepare one shared Hold’em sample

Download the checksum-pinned PHH archive, replay all 10,000 hands, and join public forecasting
and private Hero views by source ID. Both are built before reading their action labels.
Aliases follow seat rotations; memory includes only the last 64 decisions from completed hands
that the observer attended. Histories reset per directory; related suffix directories stay
in the same session split. Seed 42 assigns 80% of session groups to training before sampling.
No server outputs or prior data-sources notebook run are required.


In [ ]:
from IPython.display import FileLink, Markdown, display

main(["download-data", *COMMON])
main(["prepare", *COMMON])
display(Markdown((OUT / "example.md").read_text()))

## 2. Compare public-only and history-aware forecasts

Reuse `Forecaster(kind="masked_logistic", c=1.0)` and the Leduc `history_summary` function.
Only the history block differs. Summaries match the street, legal menu, and whether a bet
is faced; no private cards, player aliases, or source IDs enter numeric features. Keep
PokerKit’s actual legal menus, including folding when checking is free. Evaluate the same
64 validation decisions by accuracy and log loss; no hyperparameter search.


In [ ]:
main(["forecast", *COMMON])

## 3. Check summaries in the existing transformer

Download and verify the published trained BERT Mini linear checkpoint. Keep Hero’s own cards,
public state, candidates, order, and target identical. The two contexts differ only in the
appended opponent-profile section; empty history is explicit. Every train/validation input
must fit the model’s token budget without truncation and return valid candidate probabilities.
The question asks what action the player will take, because these are observed-action labels.


In [ ]:
main(["download-model", *COMMON])
main(["check", *COMMON])

## 4. Fine-tune the pair and inspect the results

Two separate loads of the same published checkpoint, one epoch each on the same 128 examples,
seed 42, batch 16, encoder LR 2e-5, head LR 1e-3, weight decay 0.01. CPU is supported.
Reuse `train_epoch`, `predict_encoded`, and the existing checkpoint format. Score local
candidate indices; verify that saving and reloading preserves probabilities and tokenization.

The injected feature is the opponent **history summary**, not learned forecaster probabilities.
This connects the two experiments through the same information source and architecture.
A learned forecast-to-decision pipeline would require a separate experiment.


In [ ]:
main(["compare", *COMMON])
main(["report", *COMMON])
display(Markdown((OUT / "results.md").read_text()))
display(FileLink(str(OUT / "explorer.html")))

Outputs include the source/split manifest, shared sample, one readable example, numeric
forecasters, both comparison tables, paired validation probabilities, and two reload-verified
fine-tuned checkpoints. `explorer.html` opens locally and shows each validation decision.
Download the output folder before a Colab runtime ends. These small, inspected validation
scores are descriptive; they do not establish reliable improvement or poker profit.

The [integration guide](../docs/meeting5-integration.md) records submission choices
and reproduction. This notebook is the initial 128/64 check. The
[CPU learning curve](../docs/meeting5-scaling.md) expands training to 4,096 examples
on fixed 1,024-example validation. The separate
[GPU ablation CLI](../docs/meeting5-ablations.md) compares encoder/head, history,
format, assignment and training settings; it does not require another notebook.
All 22 GPU fits are complete, with conditional gains that vary by seed and metric.
Use the [results index](../docs/results/README.md) for the experiment sequence and
[latest tables](../docs/meeting5-ablation-results.md) for the three-seed comparisons.
